<a href="https://colab.research.google.com/github/andreaprifti19/SINDy-Teaching-Usecases/blob/main/notebooks/02_lorenz_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Use Case 2: Discovering the Equations of the Lorenz System with SINDy

In Use Case 1 we let SINDy find the equations of a damped oscillator. That system was simple: two variables, only linear terms, and a smooth spiral towards rest.

Now we take the next step. The **Lorenz system** has three variables, **nonlinear** terms, and it is **chaotic**: its motion never repeats and never settles down. It is the classic test system of the original SINDy paper (Brunton et al., 2016), and it shows clearly where equation discovery becomes hard.

We use exactly the same workflow as in Use Case 1, so you can compare the two notebooks step by step.

**What is new compared to Use Case 1**

- Three variables and nonlinear terms ($xz$, $xy$), so the candidate library must contain products of variables.
- Chaos: a correct model still drifts away from the data after some time. We need new ways to check if a model is correct.
- A new tool, **Ensemble-SINDy**, which tells us how much we can trust each discovered term.

**What you will learn**

1. Why the Lorenz system is a harder test for SINDy than the oscillator.
2. How to run SINDy on a nonlinear system with three variables.
3. How to validate a model of a chaotic system, when comparing curves point by point no longer works.
4. When SINDy fails: noise, too much smoothing, too little data, or an unsuitable library.

**How to use this notebook**

- Run the cells **from top to bottom**, in order. Later cells use variables from earlier cells.
- If you change something, use *Runtime → Restart and run all* so that every result comes from the same run.
- All randomness uses a fixed seed, so you should see exactly the same numbers as written in the text.

**Contents**

1. Setup
2. The physical system
3. Generating the data
4. Discovering the equations with SINDy
5. Validation: is the discovered model correct?
6. Robustness experiments: noise, smoothing, Ensemble-SINDy, data quantity, candidate library
7. Summary
8. Exercises

## 1. Setup

We install the same PySINDy version as in Use Case 1 (`2.1.0`), so both notebooks behave the same.

In [1]:
!pip install pysindy==2.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.4/123.4 kB 1.3 MB/s eta 0:00:00


The imports and the reproducibility settings are the same as in Use Case 1, with one addition:

- **Warnings.** We hide the same two numerical warnings as before. In addition, with very noisy data STLSQ sometimes removes *every* term and then prints a warning. In our tables such a fit simply counts as a failure, so we hide this one message too. All other warnings are still shown.
- **Random seed.** `SEED = 0` makes the random noise identical on every run and every computer. (Section 6 explains how every noisy dataset is built from this seed.)

In [2]:
import warnings

import numpy as np
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.linalg import LinAlgWarning
from sklearn.metrics import mean_absolute_error
from sklearn.exceptions import ConvergenceWarning
import pysindy as ps

# Same two warnings as in Use Case 1 (they appear often in the robustness sweeps).
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
# Very noisy fits sometimes lose every term. We count those as failures,
# so the warning itself is not needed.
warnings.filterwarnings("ignore", message="Sparsity parameter is too big")

# Fixed seed -> identical noise, identical numbers, on every run and every machine.
SEED = 0

print("pysindy      ", ps.__version__)
print("numpy        ", np.__version__)
print("scipy        ", scipy.__version__)
print("scikit-learn ", sklearn.__version__)
print("matplotlib   ", matplotlib.__version__)

pysindy       2.1.0
numpy         2.1.3
scipy         1.16.3
scikit-learn  1.6.1
matplotlib    3.10.0


All figures are saved into the same `figures/` folder as in Use Case 1. The file names start with `lorenz_`, so they do not overwrite the figures of the first notebook.

In [3]:
import os
# In Colab: saves to "figures/". Run locally from the repo's notebooks/ folder: saves to the repo's figures/.
FIG_DIR = "../figures" if os.path.isdir("../figures") else "figures"
os.makedirs(FIG_DIR, exist_ok=True)

## 2. The physical system

In 1963, the meteorologist Edward Lorenz built a strongly simplified model of **convection**: a layer of fluid that is heated from below and cooled from above. Warm fluid rises, cool fluid sinks, and rolls of circulating fluid form. Lorenz reduced this to three variables:

- $x$: how fast the convection rolls turn (the sign gives the direction),
- $y$: the temperature difference between the rising and the sinking fluid,
- $z$: how much the vertical temperature profile differs from a straight line.

The equations are

$$\dot{x} = \sigma\,(y - x)$$
$$\dot{y} = x\,(\rho - z) - y$$
$$\dot{z} = x\,y - \beta\,z$$

**No rewriting needed.** In Use Case 1 we first had to turn a second-order equation into two first-order equations. The Lorenz system is already first-order: each variable has exactly one time derivative on the left side. So it is directly in the form that SINDy needs.

**Nonlinear terms.** The products $x\,z$ (inside $x(\rho - z)$) and $x\,y$ make the system nonlinear. They are the reason why Lorenz is chaotic, and they are the reason why our library needs terms of degree 2. For the oscillator, degree 1 would have been enough.

**Parameters.** We use the classic values from Lorenz (1963), which are also used in the SINDy paper:

- $\sigma = 10$ (Prandtl number: how fast heat spreads compared to motion),
- $\rho = 28$ (Rayleigh number, scaled: how strongly the layer is heated),
- $\beta = 8/3$ (a geometry factor of the convection rolls).

For these values the system is **chaotic**. It is completely deterministic (no randomness at all), but two starting points that are extremely close drift apart exponentially fast. This is the famous *butterfly effect*. We will see it directly in Section 5.

**The ground truth.** As in Use Case 1, `TRUE_COEF` writes the correct answer as a table. With a degree-2 library and three variables there are 10 candidate terms per equation:

| | $1$ | $x$ | $y$ | $z$ | $x^2$ | $xy$ | $xz$ | $y^2$ | $yz$ | $z^2$ |
|---|---|---|---|---|---|---|---|---|---|---|
| $\dot{x}$ | 0 | −10 | 10 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| $\dot{y}$ | 0 | 28 | −1 | 0 | 0 | 0 | −1 | 0 | 0 | 0 |
| $\dot{z}$ | 0 | 0 | 0 | −8/3 | 0 | 1 | 0 | 0 | 0 | 0 |

Only 7 of the 30 entries are non-zero, so the system is again **sparse**. Note two things for later:

- The coefficients have very different sizes: from 1 up to 28.
- The **smallest** true coefficient is 1 (the $-y$ term and the $-xz$ term in $\dot{y}$, and the $xy$ term in $\dot{z}$). This matters when we choose the threshold.

In [4]:
sigma = 10.0       # Prandtl number
rho   = 28.0       # (scaled) Rayleigh number
beta  = 8.0 / 3.0  # geometry factor

def lorenz(t, state):
    x, y, z = state
    return [sigma * (y - x),
            x * (rho - z) - y,
            x * y - beta * z]

# The answer we want SINDy to recover, written as a coefficient matrix.
# Rows = equations (x', y', z'); columns = library terms
#             1      x      y      z    x^2   x*y   x*z   y^2   y*z   z^2
TRUE_COEF = np.array([
    [0.0, -sigma, sigma,   0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0],   # x' = sigma (y - x)
    [0.0,    rho,  -1.0,   0.0,  0.0,  0.0, -1.0,  0.0,  0.0,  0.0],   # y' = rho x - y - x z
    [0.0,    0.0,   0.0, -beta,  0.0,  1.0,  0.0,  0.0,  0.0,  0.0],   # z' = x y - beta z
])

## 3. Generating the data

We again create the measurements by simulation. The settings are the same as in the Lorenz example of the PySINDy documentation, so you can compare with it:

- `dt = 0.002`: one measurement every 0.002 time units. This is 5 times finer than in Use Case 1, because the Lorenz system changes much faster.
- `t`: 10 time units, that is 5000 time points.
- `x0 = [-8, 8, 27]`: the starting point.
- `solve_ivp` with strict tolerances (`rtol`, `atol`) as in Use Case 1, so that the numerical error of the simulation is negligible.

The Lorenz equations have no physical unit of time, so we simply speak of *time units*.

Because we will simulate the Lorenz system several times (also from other starting points), we put the solver call into a small function `simulate_lorenz`. As before, `.T` gives the data matrix the shape PySINDy expects: one **row per time point**, one **column per variable**.

In [5]:
dt = 0.002                       # sampling interval
t = np.arange(0, 10, dt)         # 10 time units, 5000 samples
x0 = [-8.0, 8.0, 27.0]           # starting point

def simulate_lorenz(x0, t):
    '''Simulate the true Lorenz system; returns an array of shape (len(t), 3).'''
    sol = solve_ivp(lorenz, (t[0], t[-1]), x0, t_eval=t, rtol=1e-10, atol=1e-12)
    return sol.y.T

X = simulate_lorenz(x0, t)       # column 0 = x, column 1 = y, column 2 = z
print("Data matrix shape:", X.shape)

Data matrix shape: (5000, 3)
